# Day 2: Duplicate and leakage analysis (Tanzania maize leaves)
Run top to bottom. Use **Runtime > Change runtime type > T4 GPU** if you like (faster embeddings), but CPU works too.

What this does: embeds every image with ImageNet ResNet18, finds all pairs of very similar images, groups them into clusters, and shows you the evidence so **you** pick the similarity threshold by eye (step 5). It also checks the cross-class pairs from Day 1, the odd-size images, and estimates how much a plain random split would leak.

It expects the Day 1 data in `/content/maize_data`. If the runtime was reset, step 1 downloads it again.

## 1. Setup and data

In [ ]:
import os, glob, zipfile, subprocess, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

DATA = "/content/maize_data"      # where Day 1 unzipped the dataset
OUT = "/content/day2_out"
os.makedirs(OUT, exist_ok=True)

if not os.path.isdir(DATA):       # runtime was reset since Day 1: fetch again
    os.makedirs("/content/maize_zip", exist_ok=True)
    zp = "/content/maize_zip/maize.zip"
    subprocess.run(["wget", "-q", "-O", zp, "https://data.mendeley.com/public-api/zip/fkw49mz3xs/download/1"], check=True)
    os.makedirs(DATA, exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(DATA)
    for _ in range(3):
        inner = glob.glob(DATA + "/**/*.zip", recursive=True)
        if not inner:
            break
        for p in inner:
            with zipfile.ZipFile(p) as z:
                z.extractall(p[:-4])
            os.remove(p)
print("data folder ready")


## 2. Index the images

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png"}
paths = sorted(p for p in glob.glob(DATA + "/**/*", recursive=True) if os.path.splitext(p)[1].lower() in IMG_EXT)
df = pd.DataFrame({"path": paths})
df["cls"] = df["path"].map(lambda p: os.path.basename(os.path.dirname(p)))
df["name"] = df["path"].map(lambda p: os.path.splitext(os.path.basename(p))[0])

def get_size(p):
    with Image.open(p) as im:
        return im.size
sz = [get_size(p) for p in tqdm(paths, desc="sizes")]
df["w"] = [s[0] for s in sz]
df["h"] = [s[1] for s in sz]
df["std_size"] = [(w, h) in {(1536, 2048), (2048, 1536)} for w, h in zip(df["w"], df["h"])]
N = len(df)
print(N, "images")
print(df["cls"].value_counts())
print("standard-size images:", int(df["std_size"].sum()), "| other sizes:", int((~df["std_size"]).sum()))


## 3. Embed every image (the only step that needs PyTorch)
ResNet18 with ImageNet weights, one 512-number vector per image. On a T4 this takes a few minutes; on CPU expect longer (most of the time is decoding the big JPEGs). The vectors are mean-centred and length-normalised so that cosine similarity separates images better.

In [ ]:
# EMBED
import torch, torchvision
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class DS(torch.utils.data.Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        im = Image.open(self.paths[i])
        im.draft("RGB", (448, 448))                 # fast JPEG decode at reduced size
        im = im.convert("RGB").resize((224, 224), Image.BILINEAR)
        x = torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1).float() / 255
        return (x - MEAN) / STD

model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
model.fc = torch.nn.Identity()
model.eval().to(device)
dl = torch.utils.data.DataLoader(DS(paths), batch_size=128, num_workers=2)
feats = []
with torch.no_grad():
    for x in tqdm(dl, desc="embedding"):
        feats.append(model(x.to(device)).cpu())
F = torch.cat(feats).numpy().astype(np.float32)
F = F - F.mean(0)
F /= (np.linalg.norm(F, axis=1, keepdims=True) + 1e-8)
np.save(OUT + "/emb.npy", F.astype(np.float16))
print("embeddings:", F.shape)


## 4. Find similar pairs
Compares every image with every other image (about 44 million pairs, a minute or two) and keeps pairs with cosine similarity of at least `MIN_SIM`.

In [ ]:
MIN_SIM = 0.85
nn_sim = np.zeros(N, np.float32)
nn_idx = np.zeros(N, np.int64)
rows = []
col = np.arange(N)[None, :]
for s in tqdm(range(0, N, 1024), desc="similarity"):
    e = min(s + 1024, N)
    S = F[s:e] @ F.T
    r = np.arange(s, e)[:, None]
    S[col == r] = -1.0                      # ignore self
    nn_idx[s:e] = S.argmax(1)
    nn_sim[s:e] = S.max(1)
    S[col < r] = -1.0                       # keep each pair once
    ii, jj = np.nonzero(S >= MIN_SIM)
    rows.append(np.stack([ii + s, jj, S[ii, jj]], 1))
P = np.concatenate(rows)
P = pd.DataFrame({"i": P[:, 0].astype(int), "j": P[:, 1].astype(int), "sim": P[:, 2]})
print(len(P), "pairs with similarity >=", MIN_SIM)
if len(P) > 2_000_000:
    print("WARNING: very many pairs; raise MIN_SIM to 0.90 and re-run this cell")
plt.hist(nn_sim, bins=60)
plt.xlabel("cosine similarity to nearest other image"); plt.ylabel("images"); plt.title("How close is each image's nearest neighbour?")
plt.show()
print(pd.Series(nn_sim).describe(percentiles=[.5, .75, .9, .95, .99]).round(3))


## 5. Look at the evidence, then choose the threshold
Each band below shows random pairs from that similarity range. Find the **lowest band where the pairs still look like the same leaf or scene** (same plant, same lesion pattern, just a slightly different shot). Set `THRESH` in the next cell to the bottom of that band. If even the top band looks like different leaves, say so; the embedding is not discriminating and we need a different approach.

In [ ]:
def thumb(p, s=256):
    im = Image.open(p)
    im.draft("RGB", (s * 2, s * 2))
    im = im.convert("RGB")
    im.thumbnail((s, s))
    return im

def show_pairs(Q, title, n=4, top=False, seed=0):
    if len(Q) == 0:
        print(title, ": none"); return
    Q = Q.head(n) if top else Q.sample(min(n, len(Q)), random_state=seed)
    fig, ax = plt.subplots(len(Q), 2, figsize=(6, 2.8 * len(Q)), squeeze=False)
    for r, (_, row) in enumerate(Q.iterrows()):
        for c, k in enumerate(["i", "j"]):
            a = df.iloc[int(row[k])]
            ax[r][c].imshow(thumb(a["path"])); ax[r][c].axis("off")
            ax[r][c].set_title((f'{a["name"]} ({a["cls"]})\nsim={row["sim"]:.3f}' if c == 0 else f'{a["name"]} ({a["cls"]})'), fontsize=8)
    plt.suptitle(title); plt.tight_layout(); plt.show()

bands = [(0.85, 0.90), (0.90, 0.95), (0.95, 0.98), (0.98, 1.01)]
for lo, hi in bands:
    B = P[(P.sim >= lo) & (P.sim < hi)]
    print(f"similarity {lo:.2f} to {min(hi, 1.0):.2f}: {len(B)} pairs")
    show_pairs(B, f"similarity {lo:.2f} to {min(hi, 1.0):.2f}", n=4)


In [ ]:
THRESH = 0.90   # <-- SET THIS after looking at the galleries above (this default is a guess)

# the three Day 1 images that sat within hash distance 3 of each other across classes
named = ["MLN_83", "HEALTHY_2120", "MSV_1094"]
idx = [int(df.index[df["name"] == n][0]) for n in named if (df["name"] == n).any()]
if idx:
    fig, ax = plt.subplots(1, len(idx), figsize=(4 * len(idx), 4.2), squeeze=False)
    for k, i in enumerate(idx):
        t = thumb(df["path"][i])
        ax[0][k].imshow(t); ax[0][k].axis("off")
        ax[0][k].set_title(f'{df["name"][i]}  {df["w"][i]}x{df["h"][i]}\npixel std={np.asarray(t).std():.0f}', fontsize=8)
    plt.show()
    nm = [df["name"][i] for i in idx]
    print(pd.DataFrame(F[idx] @ F[idx].T, index=nm, columns=nm).round(3))

same = df["cls"].values[P.i.values] == df["cls"].values[P.j.values]
cc = P[(P.sim >= THRESH) & (~same)].sort_values("sim", ascending=False)
print(len(cc), "cross-class pairs at THRESH", THRESH)
show_pairs(cc, "Cross-class pairs (most similar first)", n=6, top=True)


## 6. Clusters and how much a random split would leak

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

R = {"N": N, "THRESH": THRESH}
M = P[P.sim >= THRESH]
g = coo_matrix((np.ones(len(M)), (M.i.values, M.j.values)), shape=(N, N))
nc, lab = connected_components(g, directed=False)
df["cluster"] = lab
df["cluster_size"] = df.groupby("cluster")["cluster"].transform("size")
multi = df[df.cluster_size > 1]
sizes = df.groupby("cluster").size().sort_values(ascending=False)

R["pairs"] = len(M)
R["clusters"] = int(nc)
R["multi_clusters"] = int(multi["cluster"].nunique())
R["imgs_in_multi"] = len(multi)
R["largest"] = int(sizes.iloc[0])
R["cross_clusters"] = int((multi.groupby("cluster")["cls"].nunique() > 1).sum())

print(f"pairs at threshold {THRESH}: {R['pairs']}")
print(f"clusters: {nc} total, {R['multi_clusters']} with 2+ images")
print(f"images in a multi-image cluster: {R['imgs_in_multi']} ({100 * R['imgs_in_multi'] / N:.1f}%)")
print("largest cluster sizes:", sizes.head(10).tolist())
print(f"clusters that mix classes: {R['cross_clusters']}")
print("share of each class that sits in a multi-image cluster (%):")
print((df.assign(m=df.cluster_size > 1).groupby("cls")["m"].mean() * 100).round(1))
if R["largest"] > 50:
    print("WARNING: a cluster this big usually means unrelated images got chained together; raise THRESH and re-run this cell")

def show_cluster(c, k=8):
    m = df[df.cluster == c].head(k)
    fig, ax = plt.subplots(1, len(m), figsize=(2.2 * len(m), 2.8), squeeze=False)
    for a, (_, r) in zip(ax[0], m.iterrows()):
        a.imshow(thumb(r["path"], 200)); a.axis("off"); a.set_title(r["name"], fontsize=7)
    full = df[df.cluster == c]
    plt.suptitle(f"cluster {c}: {len(full)} images, classes {sorted(full['cls'].unique())}", fontsize=9)
    plt.show()
for c in sizes.index[:3]:
    if sizes[c] > 1:
        show_cluster(c)

rng = np.random.default_rng(0)
leak = []
for _ in range(20):
    perm = rng.permutation(N)
    te = perm[: int(0.1 * N)]
    tr = set(lab[perm[int(0.1 * N): int(0.9 * N)]].tolist())
    leak.append(np.mean([int(lab[t]) in tr for t in te]))
R["leak"] = float(np.mean(leak))
print(f"\nWith a plain random 80/10/10 split, about {100 * R['leak']:.1f}% of test images would have a near-duplicate in train.")


## 7. The odd-size images
Day 1 showed ~6,600 images at exactly 1536x2048 / 2048x1536 and ~2,700 at thousands of different sizes. If the odd ones are crops of the standard ones, they are leakage candidates. Compare how close odd images are to standard images versus how close standard images are to each other.

In [ ]:
print(pd.crosstab(df["cls"], df["std_size"].map({True: "standard", False: "other size"})))
odd = np.where(~df["std_size"].values)[0]
std = np.where(df["std_size"].values)[0]
if len(odd) and len(std):
    plt.hist((df["w"] / df["h"]).values[odd], bins=50)
    plt.xlabel("width / height of odd-size images"); plt.ylabel("images"); plt.show()

    So = F[odd] @ F[std].T
    odd_nn = So.max(1); odd_to_std = std[So.argmax(1)]
    rs = np.random.default_rng(0).choice(std, min(2000, len(std)), replace=False)
    Sb = F[rs] @ F[std].T
    Sb[np.arange(len(rs)), np.searchsorted(std, rs)] = -1.0     # ignore self
    base_nn = Sb.max(1)
    cmp = pd.DataFrame({"odd -> closest standard": pd.Series(odd_nn).describe(percentiles=[.5, .9]),
                        "standard -> closest other standard": pd.Series(base_nn).describe(percentiles=[.5, .9])}).round(3)
    print(cmp)
    R["odd_n"] = len(odd)
    R["odd_median"] = float(np.median(odd_nn)); R["std_median"] = float(np.median(base_nn))
    R["odd_frac"] = float((odd_nn >= THRESH).mean())
    print(f"odd-size images whose closest standard image is >= THRESH: {100 * R['odd_frac']:.1f}%")
    Q = pd.DataFrame({"i": odd, "j": odd_to_std, "sim": odd_nn}).sort_values("sim", ascending=False)
    show_pairs(Q, "Odd-size image (left) and its closest standard image (right): most similar", n=5, top=True)
    show_pairs(Q, "Odd-size image and closest standard image: random", n=5)
else:
    print("No odd/standard split to compare.")


## 8. The `_#_#` file names
Day 1 found 13 files named like `HEALTHY_12_1`. Are they copies of the base image?

In [ ]:
sus = df[df["name"].str.match(r"^[A-Za-z]+_\d+_\d+$")]
out = []
for k, r in sus.iterrows():
    base = re.sub(r"_\d+$", "", r["name"])
    b = df.index[(df["name"] == base) & (df["cls"] == r["cls"])]
    out.append({"name": r["name"], "nearest": df["name"][nn_idx[k]], "nn_sim": round(float(nn_sim[k]), 3),
                "base_exists": len(b) > 0, "sim_to_base": round(float(F[k] @ F[b[0]]), 3) if len(b) else None})
print(pd.DataFrame(out).to_string(index=False) if out else "no such files")


## 9. Save results
`day2_clusters.csv` is what the Day 3 split will use, so keep it (the download below, or copy it to Drive). `day2_summary.md` is what to paste back into the chat.

In [ ]:
df.drop(columns=["path"]).assign(path=df["path"].str.replace(DATA, "", regex=False)).to_csv(OUT + "/day2_clusters.csv", index=False)

lines = [
    "# Day 2 summary",
    f"- images: {R['N']}; threshold used: {R['THRESH']} (pairs kept down to {MIN_SIM})",
    f"- pairs at threshold: {R['pairs']}; clusters with 2+ images: {R['multi_clusters']}; images in them: {R['imgs_in_multi']} ({100 * R['imgs_in_multi'] / R['N']:.1f}%)",
    f"- largest cluster: {R['largest']}; clusters mixing classes: {R['cross_clusters']}",
    f"- random 80/10/10 split would put a near-duplicate of {100 * R['leak']:.1f}% of test images in train",
]
if "odd_n" in R:
    lines.append(f"- odd-size images: {R['odd_n']}; median similarity to closest standard image {R['odd_median']:.3f} vs standard-to-standard {R['std_median']:.3f}; {100 * R['odd_frac']:.1f}% at or above threshold")
lines.append("- notes from the galleries (fill in): which band looked like the same leaf? are the cross-class pairs real label conflicts?")
open(OUT + "/day2_summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
try:
    from google.colab import files
    files.download(OUT + "/day2_clusters.csv")
    files.download(OUT + "/day2_summary.md")
except ImportError:
    pass
